In [2]:
# Assignment 4: E-mail (Spam) Filtering using Text Classification
#Aim: Classify messages into spam and ham using Multinomial Naive Bayes and Linear Support Vector Machine (SVM).

In [5]:
# Install required libraries


import os
import urllib.request
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Download dataset automatically if not present
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
if not os.path.exists('sms.tsv'):
    urllib.request.urlretrieve(url, 'sms.tsv')

# Load dataset
df = pd.read_csv('sms.tsv', sep='\t', header=None, names=['label', 'text'])
df['y'] = (df['label'] == 'spam').astype(int)
print("Dataset size:", df.shape[0], "| spam:", df['y'].sum(), "| ham:", (1 - df['y']).sum())

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['y'], test_size=0.2, random_state=42, stratify=df['y']
)

# TF-IDF Feature Extraction
vec = TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2), min_df=2)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)
print("TF-IDF feature matrix shape:", Xtr.shape)

models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM': LinearSVC()
}

for name, clf in models.items():
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    print(f"\n== {name} ==")
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Precision:", round(precision_score(y_test, pred), 4))
    print("Recall:", round(recall_score(y_test, pred), 4))
    print("F1-score:", round(f1_score(y_test, pred), 4))
    print("Confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(y_test, pred))

# Predict on new unseen messages
nb = models['Multinomial Naive Bayes']
new_msgs = [
    "WINNER!! You have won a free prize. Call 09061701461 now to claim",
    "Hi, are we still meeting for the project discussion tomorrow?",
    "Congratulations! Click the link to claim your FREE gift voucher today"
]

print("\nPredictions on new messages:")
for m, p in zip(new_msgs, nb.predict(vec.transform(new_msgs))):
    print(f"[{'SPAM' if p else 'HAM'}] {m}")

Dataset size: 5572 | spam: 747 | ham: 4825
TF-IDF feature matrix shape: (4457, 7369)

== Multinomial Naive Bayes ==
Accuracy: 0.9821
Precision: 0.9778
Recall: 0.8859
F1-score: 0.9296
Confusion matrix [[TN FP] [FN TP]]:
 [[963   3]
 [ 17 132]]

== Linear SVM ==
Accuracy: 0.9803
Precision: 0.9847
Recall: 0.8658
F1-score: 0.9214
Confusion matrix [[TN FP] [FN TP]]:
 [[964   2]
 [ 20 129]]

Predictions on new messages:
[SPAM] WINNER!! You have won a free prize. Call 09061701461 now to claim
[HAM] Hi, are we still meeting for the project discussion tomorrow?
[SPAM] Congratulations! Click the link to claim your FREE gift voucher today
